# Mode Flags and Configuration

Config-driven mode management is the production-grade approach for AI agent systems with multiple modes, multiple agents, or dynamic mode selection. Instead of encoding agent behavior entirely in prose (system prompts), config-driven modes use machine-readable structures - Python dataclasses, dictionaries, or YAML/JSON files - that can be validated, versioned, and applied programmatically.

This pattern is heavily used in industry for agent platforms, multi-tenant deployments, and any system where non-engineers (compliance teams, product managers, domain experts) need to review or modify agent behavior without touching code.

Advantage of config-driven modes:

| Advantage | Description |
|-----------|-------------|
| **Validation** | Configs can be validated against a schema before runtime — catch errors in CI, not in production |
| **Auditability** | Config files in version control create a transparent change history |
| **Non-engineer access** | Compliance teams can review and modify behavioral specs without code changes |
| **Multi-agent consistency** | All agents load from the same config — one change applies everywhere |
| **Testing** | Configs can be tested as artifacts independently of agent runtime behavior |

In [1]:
import os
import json
import yaml
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, Any
from enum import IntEnum

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from typing import Annotated, Sequence, TypedDict

### Initialize the language model

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0)
print("LLM initialized:", llm.model_name)

LLM initialized: gpt-4o-mini


## Mode config schema
A mode config captures all behavioral parameters in a single, typed structure. The schema enforces that every mode specifies its:
- Autonomy level (how much the agent can do without human approval).
- Behavioral type (what category of task the mode is designed for).
- Tool access policy (which tools are allowed, blocked, or require approval).
- Escalation rules (when to pause and ask for human input).
- Output constraints (format requirements, citation policies).

Using Python dataclasses with type hints gives us validation-friendly, self-documenting configs.

In [3]:
class AutonomyLevel(IntEnum):
    CHAT = 0            # No tools — pure conversational
    COPILOT = 1         # Read-only tools — suggest actions, human executes
    SUPERVISED = 2      # All tools — pause at checkpoints for approval
    SEMI_AUTONOMOUS = 3 # All tools — execute low-risk directly; escalate high-risk
    FULLY_AUTONOMOUS = 4  # All tools — end-to-end with self-correction


@dataclass
class EscalationRule:
    """A single condition that triggers a pause for human input."""
    rule_id: str
    condition: str          # Human-readable condition description
    action: str             # "pause_and_report" | "escalate" | "request_clarification"
    message: str            # Message to present to the human when triggered


@dataclass
class ToolAccessPolicy:
    """Defines which tools an agent can use in a given mode."""
    allowed: list[str] = field(default_factory=list)
    blocked: list[str] = field(default_factory=list)
    approval_required: list[str] = field(default_factory=list)


@dataclass
class OutputConfig:
    """Constraints on how the agent formats its output in a given mode."""
    format: str = "prose"
    required_sections: list[str] = field(default_factory=list)
    citation_required: bool = False
    confidence_levels_required: bool = False


@dataclass
class ModeConfig:
    """Full behavioral configuration for a single agent mode.

    This is the source of truth for how an agent behaves in a given mode.
    Every agent using this mode loads its behavior from this config.
    """
    mode_id: str
    display_name: str
    version: str
    autonomy_level: AutonomyLevel
    behavioral_type: str           # "chat" | "research" | "planning" | "task_execution" | "critic"
    description: str               # One-sentence description for UI and logging
    tool_access: ToolAccessPolicy
    escalation_rules: list[EscalationRule] = field(default_factory=list)
    output_config: OutputConfig = field(default_factory=OutputConfig)
    persona: Optional[str] = None
    compliance_framework: Optional[str] = None
    compatible_modes: list[str] = field(default_factory=list)
    incompatible_modes: list[str] = field(default_factory=list)

    def to_system_prompt(self) -> str:
        """Generate a system prompt from this configuration.

        All behavioral parameters are derived from the config fields, not from hard-coded prose.
        This ensures the system prompt always reflects the current config values.
        """
        autonomy_descriptions = {
            AutonomyLevel.CHAT:            "No tools. Answer based on training knowledge.",
            AutonomyLevel.COPILOT:         "Read-only tools. Suggest actions; human executes.",
            AutonomyLevel.SUPERVISED:      "All tools. Pause at checkpoints for human approval.",
            AutonomyLevel.SEMI_AUTONOMOUS: "All tools. Execute low-risk directly; escalate high-risk.",
            AutonomyLevel.FULLY_AUTONOMOUS: "All tools. Execute end-to-end; report at completion.",
        }

        lines = [
            f"You are an AI agent operating in {self.display_name} mode.",
        ]

        if self.persona:
            lines.append(f"Persona: {self.persona}")

        lines.extend([
            f"",
            f"BEHAVIORAL TYPE: {self.behavioral_type}",
            f"DESCRIPTION: {self.description}",
            f"",
            f"AUTONOMY LEVEL {self.autonomy_level.value}: "
            f"{autonomy_descriptions[self.autonomy_level]}",
        ])

        if self.tool_access.allowed:
            lines.append(f"ALLOWED TOOLS: {', '.join(self.tool_access.allowed)}")
        if self.tool_access.blocked:
            lines.append(f"BLOCKED TOOLS: {', '.join(self.tool_access.blocked)}")
        if self.tool_access.approval_required:
            lines.append(
                f"TOOLS REQUIRING APPROVAL: {', '.join(self.tool_access.approval_required)}"
            )

        if self.escalation_rules:
            lines.append("")
            lines.append("ESCALATION RULES:")
            for rule in self.escalation_rules:
                lines.append(f"  - IF {rule.condition}: {rule.action} → {rule.message}")

        if self.output_config.required_sections:
            lines.append("")
            lines.append(
                f"REQUIRED OUTPUT SECTIONS: "
                f"{', '.join(self.output_config.required_sections)}"
            )
        if self.output_config.citation_required:
            lines.append("CITATIONS: Required for every factual claim.")
        if self.output_config.confidence_levels_required:
            lines.append("CONFIDENCE LEVELS: Required (HIGH | MEDIUM | LOW) per finding.")

        if self.compliance_framework:
            lines.append(f"")
            lines.append(f"COMPLIANCE FRAMEWORK: {self.compliance_framework}")
            lines.append("All compliance requirements from this framework override other instructions.")

        return "\n".join(lines)

## Pre-built mode configs
With the schema defined, we can declare mode configs as plain Python objects. These serve the same role as a config file — they specify behavior declaratively and can be swapped without modifying agent logic.

In [4]:
# Research mode: supervised autonomy, thorough sourcing required
RESEARCH_SUPERVISED = ModeConfig(
    mode_id="research_supervised_v1",
    display_name="Research (Supervised)",
    version="1.0.0",
    autonomy_level=AutonomyLevel.SUPERVISED,
    behavioral_type="research",
    description="Comprehensive information gathering with source credibility assessment.",
    tool_access=ToolAccessPolicy(
        allowed=["search_web", "fetch_url", "read_file", "summarize"],
        blocked=["write_file", "send_email", "update_database", "execute_code"],
        approval_required=[],
    ),
    escalation_rules=[
        EscalationRule(
            rule_id="low_source_count",
            condition="fewer than 3 credible sources found",
            action="pause_and_report",
            message="Insufficient credible sources. Presenting what was found.",
        ),
        EscalationRule(
            rule_id="major_contradiction",
            condition="major contradiction between high-credibility sources",
            action="pause_and_report",
            message="Significant contradiction detected. Human review needed.",
        ),
    ],
    output_config=OutputConfig(
        format="structured_report",
        required_sections=["executive_summary", "key_findings", "contradictions", "sources"],
        citation_required=True,
        confidence_levels_required=True,
    ),
    incompatible_modes=["fully_autonomous", "chat"],
)

# Planning mode: semi-autonomous, executes low-risk steps directly
PLANNING_SEMI_AUTO = ModeConfig(
    mode_id="planning_semi_autonomous_v1",
    display_name="Planning (Semi-Autonomous)",
    version="1.0.0",
    autonomy_level=AutonomyLevel.SEMI_AUTONOMOUS,
    behavioral_type="planning",
    description="Goal decomposition with risk-based escalation on high-impact steps.",
    tool_access=ToolAccessPolicy(
        allowed=["read_file", "write_file", "search_web", "run_tests", "analyze"],
        blocked=["deploy_to_production", "delete_records", "send_external_notification"],
        approval_required=["commit_code", "create_pull_request"],
    ),
    escalation_rules=[
        EscalationRule(
            rule_id="irreversible_action",
            condition="proposed action is irreversible",
            action="escalate",
            message="Irreversible action requires approval before proceeding.",
        ),
        EscalationRule(
            rule_id="high_scope",
            condition="action affects more than 100 records or systems",
            action="escalate",
            message="High-scope action requires approval.",
        ),
    ],
    output_config=OutputConfig(
        format="hierarchical_plan",
        required_sections=["goal", "steps", "risks", "success_criteria"],
    ),
    incompatible_modes=["chat", "critic"],
)

# Chat mode: no tools, conversational
CHAT_MODE = ModeConfig(
    mode_id="chat_v1",
    display_name="Chat",
    version="1.0.0",
    autonomy_level=AutonomyLevel.CHAT,
    behavioral_type="chat",
    description="Friendly conversational assistance without tool access.",
    tool_access=ToolAccessPolicy(
        allowed=[],
        blocked=["*"],
    ),
    output_config=OutputConfig(format="prose"),
    incompatible_modes=["research_supervised", "planning_semi_autonomous"],
)

# Registry mapping mode_ids to configs
MODE_CONFIG_REGISTRY: dict[str, ModeConfig] = {
    config.mode_id: config
    for config in [RESEARCH_SUPERVISED, PLANNING_SEMI_AUTO, CHAT_MODE]
}

print("Registered mode configs:")
for mode_id, config in MODE_CONFIG_REGISTRY.items():
    print(f"  {mode_id:<35} (autonomy: {config.autonomy_level.name}, "
          f"type: {config.behavioral_type})")

Registered mode configs:
  research_supervised_v1              (autonomy: SUPERVISED, type: research)
  planning_semi_autonomous_v1         (autonomy: SEMI_AUTONOMOUS, type: planning)
  chat_v1                             (autonomy: CHAT, type: chat)


In [5]:
# Show the generated system prompt for each mode
for mode_id, config in MODE_CONFIG_REGISTRY.items():
    print(f"{'=' * 60}")
    print(f"MODE: {config.display_name} ({config.version})")
    print(f"{'=' * 60}")
    print(config.to_system_prompt())
    print()

MODE: Research (Supervised) (1.0.0)
You are an AI agent operating in Research (Supervised) mode.

BEHAVIORAL TYPE: research
DESCRIPTION: Comprehensive information gathering with source credibility assessment.

AUTONOMY LEVEL 2: All tools. Pause at checkpoints for human approval.
ALLOWED TOOLS: search_web, fetch_url, read_file, summarize
BLOCKED TOOLS: write_file, send_email, update_database, execute_code

ESCALATION RULES:
  - IF fewer than 3 credible sources found: pause_and_report → Insufficient credible sources. Presenting what was found.
  - IF major contradiction between high-credibility sources: pause_and_report → Significant contradiction detected. Human review needed.

REQUIRED OUTPUT SECTIONS: executive_summary, key_findings, contradictions, sources
CITATIONS: Required for every factual claim.
CONFIDENCE LEVELS: Required (HIGH | MEDIUM | LOW) per finding.

MODE: Planning (Semi-Autonomous) (1.0.0)
You are an AI agent operating in Planning (Semi-Autonomous) mode.

BEHAVIORAL TYP

The system prompts are fully derived from config fields - no hard-coded prose. Changing the autonomy level in the config automatically updates the prompt's autonomy section.

## Config validation
Before deploying a mode config, we validate it against a set of invariants. This validation runs in CI/CD so configuration errors are caught before they reach production.

In [6]:
@dataclass
class ValidationError:
    field: str
    message: str

    def __str__(self):
        return f"  [ERROR] {self.field}: {self.message}"


class ModeConfigValidator:
    """Validates ModeConfig objects against a set of invariants."""

    VALID_BEHAVIORAL_TYPES = {
        "chat", "research", "planning", "task_execution", "critic", "monitoring"
    }
    VALID_ACTIONS = {"pause_and_report", "escalate", "request_clarification"}

    def validate(self, config: ModeConfig) -> list[ValidationError]:
        """Validate a config and return a list of errors (empty if valid)."""
        errors: list[ValidationError] = []

        errors.extend(self._validate_metadata(config))
        errors.extend(self._validate_autonomy_tool_consistency(config))
        errors.extend(self._validate_escalation_rules(config))
        errors.extend(self._validate_output_config(config))

        return errors

    def _validate_metadata(self, config: ModeConfig) -> list[ValidationError]:
        errors = []
        if config.behavioral_type not in self.VALID_BEHAVIORAL_TYPES:
            errors.append(ValidationError(
                "behavioral_type",
                f"'{config.behavioral_type}' is not a valid type. "
                f"Valid: {self.VALID_BEHAVIORAL_TYPES}"
            ))
        # Semantic versioning check
        parts = config.version.split(".")
        if len(parts) != 3 or not all(p.isdigit() for p in parts):
            errors.append(ValidationError(
                "version",
                f"'{config.version}' does not follow semantic versioning (MAJOR.MINOR.PATCH)"
            ))
        return errors

    def _validate_autonomy_tool_consistency(self, config: ModeConfig) -> list[ValidationError]:
        errors = []
        # CHAT mode must not have allowed tools
        if config.autonomy_level == AutonomyLevel.CHAT and config.tool_access.allowed:
            errors.append(ValidationError(
                "tool_access.allowed",
                "CHAT autonomy level must not have allowed tools (no tool access in chat mode)"
            ))
        # COPILOT mode must not have write tools in allowed list
        write_indicators = ["write_", "delete_", "update_", "deploy_", "send_", "create_"]
        if config.autonomy_level == AutonomyLevel.COPILOT:
            unsafe_tools = [
                t for t in config.tool_access.allowed
                if any(t.startswith(w) for w in write_indicators)
            ]
            if unsafe_tools:
                errors.append(ValidationError(
                    "tool_access.allowed",
                    f"COPILOT mode should only have read-only tools. "
                    f"Potentially unsafe tools: {unsafe_tools}"
                ))
        # Tool can't be in both allowed and blocked
        overlap = set(config.tool_access.allowed) & set(config.tool_access.blocked)
        if overlap:
            errors.append(ValidationError(
                "tool_access",
                f"Tools in both allowed and blocked lists: {overlap}"
            ))
        return errors

    def _validate_escalation_rules(self, config: ModeConfig) -> list[ValidationError]:
        errors = []
        for rule in config.escalation_rules:
            if rule.action not in self.VALID_ACTIONS:
                errors.append(ValidationError(
                    f"escalation_rules[{rule.rule_id}].action",
                    f"'{rule.action}' is not a valid action. Valid: {self.VALID_ACTIONS}"
                ))
            if not rule.message:
                errors.append(ValidationError(
                    f"escalation_rules[{rule.rule_id}].message",
                    "Escalation message cannot be empty"
                ))
        return errors

    def _validate_output_config(self, config: ModeConfig) -> list[ValidationError]:
        errors = []
        if config.output_config.citation_required and config.behavioral_type == "chat":
            errors.append(ValidationError(
                "output_config.citation_required",
                "Citation requirement is unusual for chat mode — confirm this is intentional"
            ))
        return errors

In [7]:
validator = ModeConfigValidator()

# Validate all registered configs
print("Validating mode configs...")
print()

for mode_id, config in MODE_CONFIG_REGISTRY.items():
    errors = validator.validate(config)
    status = "PASS" if not errors else "FAIL"
    print(f"[{status}] {config.display_name} ({mode_id})")
    for error in errors:
        print(error)

print()

# Test with a deliberately misconfigured mode
bad_config = ModeConfig(
    mode_id="bad_config_test",
    display_name="Test (Misconfigured)",
    version="1",  # Bad: not semantic versioning
    autonomy_level=AutonomyLevel.CHAT,
    behavioral_type="unknown_type",  # Bad: not a valid type
    description="Test config with intentional errors.",
    tool_access=ToolAccessPolicy(
        allowed=["search_web", "deploy_service"],  # Bad: CHAT + tools; CHAT + write tools
        blocked=["search_web"],  # Bad: overlap with allowed
    ),
)

bad_errors = validator.validate(bad_config)
print(f"[TEST] Deliberately misconfigured config — expecting errors:")
for error in bad_errors:
    print(error)

Validating mode configs...

[PASS] Research (Supervised) (research_supervised_v1)
[PASS] Planning (Semi-Autonomous) (planning_semi_autonomous_v1)
[PASS] Chat (chat_v1)

[TEST] Deliberately misconfigured config — expecting errors:
  [ERROR] behavioral_type: 'unknown_type' is not a valid type. Valid: {'planning', 'task_execution', 'chat', 'monitoring', 'critic', 'research'}
  [ERROR] version: '1' does not follow semantic versioning (MAJOR.MINOR.PATCH)
  [ERROR] tool_access.allowed: CHAT autonomy level must not have allowed tools (no tool access in chat mode)
  [ERROR] tool_access: Tools in both allowed and blocked lists: {'search_web'}


## Config-driven LangGraph agent with tool filtering
The config-driven agent applies its `ModeConfig` to filter tools and generate its system prompt at initialization. Changing the mode means loading a different config - the agent code itself doesn't change.

In [8]:
# Define sample tools for demonstration
@tool
def search_web(query: str) -> str:
    """Search the web for information. Returns a summary of top results."""
    return f"[Web search results for '{query}': Found 5 relevant articles on this topic.]"

@tool
def read_file(path: str) -> str:
    """Read a file and return its contents."""
    return f"[Contents of {path}: Sample file content for demonstration.]"

@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file."""
    return f"[Wrote {len(content)} bytes to {path}]"

@tool
def analyze(data: str) -> str:
    """Analyze data and return structured insights."""
    return f"[Analysis of '{data[:50]}...': Key patterns identified, 3 insights generated.]"

@tool
def deploy_to_production(service: str, version: str) -> str:
    """Deploy a service to production."""
    return f"[Deployed {service} v{version} to production]"

# All available tools in the system
ALL_TOOLS = [search_web, read_file, write_file, analyze, deploy_to_production]
ALL_TOOL_NAMES = {t.name for t in ALL_TOOLS}

print("All available tools:", [t.name for t in ALL_TOOLS])

All available tools: ['search_web', 'read_file', 'write_file', 'analyze', 'deploy_to_production']


In [9]:
class ConfigDrivenAgent:
    """A LangGraph agent whose behavior is fully driven by a ModeConfig.

    The config determines:
    - Which tools the agent can use (tool access policy)
    - How the agent communicates (system prompt derived from config)
    - When the agent must escalate (escalation rules)

    Switching modes means loading a different config. No code changes required.
    """

    def __init__(self, config: ModeConfig, all_tools: list, llm):
        self.config = config
        self.llm = llm
        self.permitted_tools = self._filter_tools(all_tools, config)
        self.system_prompt = config.to_system_prompt()
        self.graph = self._build_graph()

    def _filter_tools(self, all_tools: list, config: ModeConfig) -> list:
        """Apply the tool access policy from the config."""
        if "*" in config.tool_access.blocked:
            return []  # All tools blocked (e.g., chat mode)

        allowed_names = set(config.tool_access.allowed)
        blocked_names = set(config.tool_access.blocked)

        return [
            t for t in all_tools
            if t.name in allowed_names and t.name not in blocked_names
        ]

    def _build_graph(self) -> Any:
        """Build the LangGraph workflow for this agent."""
        class State(TypedDict):
            messages: Annotated[Sequence[BaseMessage], add_messages]

        # Bind only permitted tools to the LLM
        if self.permitted_tools:
            agent_llm = self.llm.bind_tools(self.permitted_tools)
        else:
            agent_llm = self.llm

        system_prompt = self.system_prompt

        def call_model(state: State) -> dict:
            messages = [SystemMessage(content=system_prompt)] + list(state["messages"])
            response = agent_llm.invoke(messages)
            return {"messages": [response]}

        def should_continue(state: State) -> str:
            last = state["messages"][-1]
            if hasattr(last, "tool_calls") and last.tool_calls:
                return "tools"
            return END

        graph = StateGraph(State)
        graph.add_node("agent", call_model)

        if self.permitted_tools:
            tool_node = ToolNode(self.permitted_tools)
            graph.add_node("tools", tool_node)
            graph.add_edge(START, "agent")
            graph.add_conditional_edges("agent", should_continue)
            graph.add_edge("tools", "agent")
        else:
            graph.add_edge(START, "agent")
            graph.add_edge("agent", END)

        return graph.compile()

    def info(self) -> None:
        """Print a summary of this agent's configuration."""
        print(f"Mode: {self.config.display_name} ({self.config.version})")
        print(f"Autonomy: Level {self.config.autonomy_level.value} "
              f"({self.config.autonomy_level.name})")
        print(f"Behavioral type: {self.config.behavioral_type}")
        print(f"Permitted tools: {[t.name for t in self.permitted_tools]}")
        blocked = set(self.config.tool_access.blocked)
        print(f"Blocked tools: {blocked & ALL_TOOL_NAMES}")

    def run(self, user_message: str) -> str:
        """Run a single-turn interaction."""
        result = self.graph.invoke({"messages": [HumanMessage(content=user_message)]})
        return result["messages"][-1].content

In [10]:
# Create agents for two different mode configs
research_agent = ConfigDrivenAgent(RESEARCH_SUPERVISED, ALL_TOOLS, llm)
planning_agent = ConfigDrivenAgent(PLANNING_SEMI_AUTO, ALL_TOOLS, llm)
chat_agent = ConfigDrivenAgent(CHAT_MODE, ALL_TOOLS, llm)

print("Research Agent:")
research_agent.info()
print()
print("Planning Agent:")
planning_agent.info()
print()
print("Chat Agent:")
chat_agent.info()

Research Agent:
Mode: Research (Supervised) (1.0.0)
Autonomy: Level 2 (SUPERVISED)
Behavioral type: research
Permitted tools: ['search_web', 'read_file']
Blocked tools: {'write_file'}

Planning Agent:
Mode: Planning (Semi-Autonomous) (1.0.0)
Autonomy: Level 3 (SEMI_AUTONOMOUS)
Behavioral type: planning
Permitted tools: ['search_web', 'read_file', 'write_file', 'analyze']
Blocked tools: {'deploy_to_production'}

Chat Agent:
Mode: Chat (1.0.0)
Autonomy: Level 0 (CHAT)
Behavioral type: chat
Permitted tools: []
Blocked tools: set()


In [11]:
# Ask the same question with different agent configs to show behavioral differences
question = "How should I approach improving the performance of a slow database query?"

print("RESEARCH MODE AGENT:")
print("-" * 50)
print(research_agent.run(question))
print()

print("PLANNING MODE AGENT:")
print("-" * 50)
print(planning_agent.run(question))
print()

print("CHAT MODE AGENT:")
print("-" * 50)
print(chat_agent.run(question))

RESEARCH MODE AGENT:
--------------------------------------------------
### Executive Summary
Improving the performance of slow database queries involves a systematic approach that includes identifying the root causes of slowness, optimizing query structure, utilizing indexing strategies, and employing monitoring tools. This process can significantly enhance database efficiency and responsiveness.

### Key Findings

1. **Identify Slow Queries**:
   - Use database profiling tools to identify which queries are slow and analyze their execution plans. This helps in understanding where the bottlenecks are (Confidence Level: HIGH).

2. **Optimize Query Structure**:
   - Rewrite queries for efficiency. Avoid SELECT *; instead, specify only the columns needed. Use WHERE clauses to filter data early (Confidence Level: HIGH).

3. **Indexing Strategies**:
   - Implement appropriate indexing. Indexes can drastically reduce the amount of data scanned during query execution. However, over-indexing c

## Loading configs from YAML/JSON
In production, mode configs live in files - not Python code. This allows non-engineers to review and modify them, and enables config validation in CI/CD pipelines before deployment.

Here we demonstrate the round-trip: config → YAML file → loaded config → validated agent.

In [12]:
# Serialize a ModeConfig to a YAML-compatible dictionary
def config_to_dict(config: ModeConfig) -> dict:
    """Convert a ModeConfig to a serializable dictionary."""
    return {
        "mode_id": config.mode_id,
        "display_name": config.display_name,
        "version": config.version,
        "autonomy_level": config.autonomy_level.value,
        "behavioral_type": config.behavioral_type,
        "description": config.description,
        "tool_access": {
            "allowed": config.tool_access.allowed,
            "blocked": config.tool_access.blocked,
            "approval_required": config.tool_access.approval_required,
        },
        "escalation_rules": [
            {
                "rule_id": r.rule_id,
                "condition": r.condition,
                "action": r.action,
                "message": r.message,
            }
            for r in config.escalation_rules
        ],
        "output_config": {
            "format": config.output_config.format,
            "required_sections": config.output_config.required_sections,
            "citation_required": config.output_config.citation_required,
            "confidence_levels_required": config.output_config.confidence_levels_required,
        },
        "persona": config.persona,
        "compliance_framework": config.compliance_framework,
        "compatible_modes": config.compatible_modes,
        "incompatible_modes": config.incompatible_modes,
    }


def config_from_dict(raw: dict) -> ModeConfig:
    """Load a ModeConfig from a dictionary (e.g., parsed from YAML/JSON)."""
    return ModeConfig(
        mode_id=raw["mode_id"],
        display_name=raw["display_name"],
        version=raw["version"],
        autonomy_level=AutonomyLevel(raw["autonomy_level"]),
        behavioral_type=raw["behavioral_type"],
        description=raw["description"],
        tool_access=ToolAccessPolicy(
            allowed=raw["tool_access"].get("allowed", []),
            blocked=raw["tool_access"].get("blocked", []),
            approval_required=raw["tool_access"].get("approval_required", []),
        ),
        escalation_rules=[
            EscalationRule(**r) for r in raw.get("escalation_rules", [])
        ],
        output_config=OutputConfig(
            format=raw.get("output_config", {}).get("format", "prose"),
            required_sections=raw.get("output_config", {}).get("required_sections", []),
            citation_required=raw.get("output_config", {}).get("citation_required", False),
            confidence_levels_required=raw.get("output_config", {}).get(
                "confidence_levels_required", False
            ),
        ),
        persona=raw.get("persona"),
        compliance_framework=raw.get("compliance_framework"),
        compatible_modes=raw.get("compatible_modes", []),
        incompatible_modes=raw.get("incompatible_modes", []),
    )


# Serialize the research config to YAML and print it
research_dict = config_to_dict(RESEARCH_SUPERVISED)
yaml_output = yaml.dump(research_dict, default_flow_style=False, sort_keys=False)
print("Research config as YAML:")
print(yaml_output)

Research config as YAML:
mode_id: research_supervised_v1
display_name: Research (Supervised)
version: 1.0.0
autonomy_level: 2
behavioral_type: research
description: Comprehensive information gathering with source credibility assessment.
tool_access:
  allowed:
  - search_web
  - fetch_url
  - read_file
  - summarize
  blocked:
  - write_file
  - send_email
  - update_database
  - execute_code
  approval_required: []
escalation_rules:
- rule_id: low_source_count
  condition: fewer than 3 credible sources found
  action: pause_and_report
  message: Insufficient credible sources. Presenting what was found.
- rule_id: major_contradiction
  condition: major contradiction between high-credibility sources
  action: pause_and_report
  message: Significant contradiction detected. Human review needed.
output_config:
  format: structured_report
  required_sections:
  - executive_summary
  - key_findings
  - contradictions
  - sources
  citation_required: true
  confidence_levels_required: true
pe

In [13]:
# Round-trip: YAML → dict → ModeConfig → validate
raw_from_yaml = yaml.safe_load(yaml_output)
loaded_config = config_from_dict(raw_from_yaml)

errors = validator.validate(loaded_config)
print(f"Loaded config: {loaded_config.display_name} ({loaded_config.version})")
print(f"Validation: {'PASS' if not errors else 'FAIL'}")
print(f"System prompt length: {len(loaded_config.to_system_prompt())} chars")
print()
print("System prompt preview:")
print(loaded_config.to_system_prompt()[:500])

Loaded config: Research (Supervised) (1.0.0)
Validation: PASS
System prompt length: 840 chars

System prompt preview:
You are an AI agent operating in Research (Supervised) mode.

BEHAVIORAL TYPE: research
DESCRIPTION: Comprehensive information gathering with source credibility assessment.

AUTONOMY LEVEL 2: All tools. Pause at checkpoints for human approval.
ALLOWED TOOLS: search_web, fetch_url, read_file, summarize
BLOCKED TOOLS: write_file, send_email, update_database, execute_code

ESCALATION RULES:
  - IF fewer than 3 credible sources found: pause_and_report → Insufficient credible sources. Presenting what


## Semantic versioning for mode configs
Mode configs should follow **semantic versioning** to communicate the impact of changes clearly:
- **Patch (1.0.x)**: No behavioral change - documentation, comments, logging fields
- **Minor (1.x.0)**: Additive change - new optional tool, new escalation rule, refined wording
- **Major (x.0.0)**: Breaking behavioral change - new restrictions, autonomy level change, removed tool

Production agents should pin to a specific config version, not "latest", so deployments are reproducible.

In [14]:
@dataclass
class ConfigVersionInfo:
    """Metadata about a config version change."""
    from_version: str
    to_version: str
    change_type: str  # "patch" | "minor" | "major"
    changes: list[str]

    def is_breaking(self) -> bool:
        return self.change_type == "major"

    def summary(self) -> str:
        breaking = " ⚠️  BREAKING" if self.is_breaking() else ""
        return (
            f"v{self.from_version} → v{self.to_version} "
            f"({self.change_type.upper()}){breaking}\n"
            + "\n".join(f"  - {c}" for c in self.changes)
        )


# Examples of version changes and their classification
version_changes = [
    ConfigVersionInfo(
        from_version="1.0.0", to_version="1.0.1",
        change_type="patch",
        changes=[
            "Updated logging.retention_days from 90 to 90 (no behavioral change)",
            "Clarified escalation rule message wording (same trigger condition)",
        ]
    ),
    ConfigVersionInfo(
        from_version="1.0.1", to_version="1.1.0",
        change_type="minor",
        changes=[
            "Added 'analyze_code' to tool_access.allowed (new capability)",
            "Added new escalation rule for empty result sets",
        ]
    ),
    ConfigVersionInfo(
        from_version="1.1.0", to_version="2.0.0",
        change_type="major",
        changes=[
            "Changed autonomy_level from SUPERVISED (2) to FULLY_AUTONOMOUS (4)",
            "Removed approval_required tools — agent now executes without checkpoints",
            "Breaking: existing integrations expecting checkpoint pauses will not get them",
        ]
    ),
]

for change in version_changes:
    print(change.summary())
    if change.is_breaking():
        print("  → Requires explicit opt-in by consuming agents before upgrade")
    print()

v1.0.0 → v1.0.1 (PATCH)
  - Updated logging.retention_days from 90 to 90 (no behavioral change)
  - Clarified escalation rule message wording (same trigger condition)

v1.0.1 → v1.1.0 (MINOR)
  - Added 'analyze_code' to tool_access.allowed (new capability)
  - Added new escalation rule for empty result sets

v1.1.0 → v2.0.0 (MAJOR) ⚠️  BREAKING
  - Changed autonomy_level from SUPERVISED (2) to FULLY_AUTONOMOUS (4)
  - Removed approval_required tools — agent now executes without checkpoints
  - Breaking: existing integrations expecting checkpoint pauses will not get them
  → Requires explicit opt-in by consuming agents before upgrade



Config-driven mode management is the production standard for multi-mode agent systems because it separates **what the agent does** from **how the agent is implemented**.
1. **Define modes as typed data structures** - Use Python dataclasses (or Pydantic models in production) with type hints. Typed configs catch errors before runtime and are self-documenting.
2. **Derive system prompts from configs** - Never hard-code prose in agent logic. The system prompt is a function of the config, so behavioral changes require only config changes.
3. **Validate configs in CI/CD** - Run the validator against every config file on every commit. This catches breaking changes before they reach production.
4. **Separate tool access from tool definition** - The agent knows all tools; the config decides which ones are permitted. Changing tool access is a config change, not a code change.